# Phase 3 — Segmentation Engine
Multi-tool: **Python → SQL → R**

**Requires:** Phase 1 complete (customers_snapshots.csv in Drive)

In [ ]:
!pip install openpyxl plotly psycopg2-binary sqlalchemy --quiet
!Rscript -e 'install.packages(c("jsonlite","dplyr"), repos="https://cloud.r-project.org/")' 2>/dev/null
print('Dependencies ready.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/CustomerHealthForensics'
import os, sys
os.makedirs(f'{PROJECT_ROOT}/outputs', exist_ok=True)
sys.path.insert(0, f'{PROJECT_ROOT}/src')
sys.path.insert(0, f'{PROJECT_ROOT}/src/orchestration')
print('Drive mounted.')

In [ ]:
from google.colab import files
print('Upload ALL src/ files:')
print('  segmentation_engine.py, cohort_analysis.py')
print('  db/db_connector.py, validation/r_runner.py, validation/octave_runner.py')
print('  orchestration/pipeline_runner.py')
print('Also upload: r/segmentation_analysis.R, r/drift_validation.R')
uploaded = files.upload()
for fname, data in uploaded.items():
    dest = f"{PROJECT_ROOT}/src/{fname}"
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    with open(dest, 'wb') as f: f.write(data)
    print(f'  {fname}')

In [ ]:
# Segmentation only (no SQL/R/Octave)
from segmentation_engine import build_from_snapshots
from cohort_analysis import build_all_cohort_tables, export_to_excel
import pandas as pd
from pathlib import Path

df_snap = pd.read_csv(f'{PROJECT_ROOT}/data/customers_snapshots.csv', nrows=600_000)
engine  = build_from_snapshots(df_snap, verbose=True)
results = engine.run()

print(f"Segments: {results['n_segments_analyzed']}")
print(f"Degrading: {results['global_insights']['n_degrading']}")
print(f"Revenue at risk: ${results['global_insights']['total_revenue_at_risk']:,.0f}")

In [ ]:
# Full multi-tool pipeline (Python + SQL + R)
from orchestration.pipeline_runner import run_full_pipeline

intel = run_full_pipeline(
    data_dir    = Path(f'{PROJECT_ROOT}/data'),
    output_dir  = Path(f'{PROJECT_ROOT}/outputs'),
    sample_size = 600_000,
    skip_db     = True,   # set False if PostgreSQL is available
    skip_r      = False,  # R must be installed
    skip_octave = True,   # set False for Octave validation
)

In [ ]:
# Inspect top degrading segments
import json
insights = json.load(open(f'{PROJECT_ROOT}/outputs/segmentation/global_insights.json'))
for s in insights['top_degrading_segments']:
    print(f"{s['segment_id']:30s}  churn={s['churn_rate']:.3f}  delta=+{s.get('churn_delta',0):.3f}  ${s['revenue_at_risk']:,.0f}")

In [ ]:
# Plotly cohort heatmap
import plotly.express as px, pandas as pd
tbl = pd.read_csv(f'{PROJECT_ROOT}/outputs/segmentation/cohort_tables/plan_type_x_region__churn_rate.csv', index_col=0)
px.imshow(tbl, color_continuous_scale='RdYlGn_r', title='Churn Rate: Plan × Region').show()

In [ ]:
# R ANOVA results
r_out = json.load(open(f'{PROJECT_ROOT}/outputs/r_outputs/segmentation_plan_type.json'))
anova = r_out.get('anova',{})
print(f"ANOVA: F={anova.get('f_statistic')}  p={anova.get('p_value')}  significant={anova.get('significant')}")
print(f"{anova.get('interpretation','')}")

## ✅ Phase 3 Complete

```
outputs/segmentation/  ← Python segmentation
outputs/sql_outputs/   ← SQL analytical queries
outputs/r_outputs/     ← R statistical validation
```
Next → Phase4_Drift_Colab.ipynb